In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 18.5 MB/s eta 0:00:0000:0100:01
  Created wheel for gtfparse: filename=gtfparse-2.0.1-py3-none-any.whl size=15285 sha256=26b17e6fd57b43547766eb9c7dcc75e4f1686e1e9eb29e1aa19d812748009a41
  Stored in directory: /root/.cache/pip/wheels/91/da/d4/4168bc0aa594bfcda1ba95d81ea91552d506807d686ceb4e1e
Successfully built gtfparse
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 8.4 MB/s eta 0:00:00:00:0100:01
Reason for being yanked: <none given>
  Attempting uninstall: polars
    Found existing installation: polars 0.18.4
    Uninstalling polars-0.18.4:
      Successfully uninstalled polars-0.18.4


In [2]:
!pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.1/39.1 MB 20.8 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install anndata==0.8.0

In [1]:
from samalg import SAM
from Bio import SeqIO
from gtfparse import read_gtf
import pandas as pd
import pandas
import pyarrow
import pickle
import scanpy as sc
import numpy as np

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
#open scdata to see how many gene matches you have
dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_MO_soupX_plus5.h5ad')

In [39]:
dat

AnnData object with n_obs × n_vars = 84652 × 19307
    obs: 'n_counts', 'n_genes', 'key'

In [40]:
dat.var_names

Index(['ENSMOCG00000000002', 'ENSMOCG00000000003', 'ENSMOCG00000000004',
       'ENSMOCG00000000005', 'ND1', 'ENSMOCG00000000008', 'ENSMOCG00000000009',
       'ND2', 'ENSMOCG00000000011', 'ENSMOCG00000000012',
       ...
       'ENSMOCG00000023030', 'Gemin2', 'Creb3', 'Ppfia4', 'Vps28', 'Trp53bp2',
       'Rfc1', 'ENSMOCG00000023037', 'Hbs1l', 'Celsr1'],
      dtype='object', length=19307)

In [41]:
input_file = open("../../cDNA_fasta/Microtus_ochrogaster.MicOch1.0.112.cdna.all.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    print(item)
    break

ID: ENSMOCT00000000006.1
Name: ENSMOCT00000000006.1
Description: ENSMOCT00000000006.1 cdna chromosome:MicOch1.0:MT:2728:3682:1 gene:ENSMOCG00000000006.1 gene_biotype:protein_coding transcript_biotype:protein_coding gene_symbol:ND1 description:NADH dehydrogenase subunit 1 [Source:NCBI gene;Acc:26045311]
Number of features: 0
Seq('GTGTATTTCATCAACATACTAACACTTTTAGTCCCAGTTTTAATTGCCATAGCA...CCT', SingleLetterAlphabet())


In [42]:
#pull longest gene
input_file = open("../../cDNA_fasta/Microtus_ochrogaster.MicOch1.0.112.cdna.all.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    #manually change gene_ID to match which field you would like to see in your BLAST table
    gene_ID = item.description.split('gene:')[1].split(' ')[0]
    if gene_ID in gene_dict.keys():
        if len(item.seq) > len(gene_dict[gene_ID].seq):
            gene_dict[gene_ID] = item
    else:
        gene_dict[gene_ID] = item
len(gene_dict)

19659

In [43]:
db_gene = read_gtf('../../cDNA_fasta/Microtus_ochrogaster.MicOch1.0.112.gtf', features = ['gene','transcript'])
df_gene = db_gene.to_pandas()

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_source', 'transcript_biotype', 'tag', 'gene_name', 'transcript_name']


In [44]:
#use if gene_id is not aligned between cdna and fasta ex: japanese quail
fti = []
for item in df_gene.index:
    fti.append(df_gene.loc[item,'gene_id'] + "." + df_gene.loc[item,'gene_version'])
df_gene['Full_gene_id'] = fti

In [45]:
df_gene

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_version,gene_source,gene_biotype,transcript_id,transcript_version,transcript_source,transcript_biotype,tag,gene_name,transcript_name,Full_gene_id
0,1,ensembl,gene,23719295,23734994,NaN,+,0,ENSMOCG00000000281,1,ensembl,protein_coding,,,,,,,,ENSMOCG00000000281.1
1,1,ensembl,transcript,23719295,23734994,NaN,+,0,ENSMOCG00000000281,1,ensembl,protein_coding,ENSMOCT00000000366,1,ensembl,protein_coding,Ensembl_canonical,,,ENSMOCG00000000281.1
2,1,ensembl,gene,46380509,46480297,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,,,,,,Tecpr2,,ENSMOCG00000000369.1
3,1,ensembl,transcript,46380509,46480297,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,ENSMOCT00000000489,1,ensembl,protein_coding,Ensembl_canonical,Tecpr2,Tecpr2-201,ENSMOCG00000000369.1
4,1,ensembl,transcript,46387947,46478097,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,ENSMOCT00000000494,1,ensembl,protein_coding,,Tecpr2,Tecpr2-202,ENSMOCG00000000369.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54552,AHZW01186317.1,ensembl,transcript,594,720,NaN,-,0,ENSMOCG00000006135,1,ensembl,miRNA,ENSMOCT00000007974,1,ensembl,miRNA,Ensembl_canonical,,,ENSMOCG00000006135.1
54553,AHZW01186775.1,ensembl,gene,949,1039,NaN,-,0,ENSMOCG00000005747,1,ensembl,miRNA,,,,,,,,ENSMOCG00000005747.1
54554,AHZW01186775.1,ensembl,transcript,949,1039,NaN,-,0,ENSMOCG00000005747,1,ensembl,miRNA,ENSMOCT00000007482,1,ensembl,miRNA,Ensembl_canonical,,,ENSMOCG00000005747.1
54555,AHZW01186170.1,ensembl,gene,339,436,NaN,+,0,ENSMOCG00000008661,1,ensembl,miRNA,,,,,,,,ENSMOCG00000008661.1


In [ ]:
#fin_gene_dict = {}
#for item in gene_dict.keys():
#    if len(df_gene.loc[df_gene.index[df_gene['Full_gene_id'] == item][0], 'gene_name']) > 0:
#        fin_gene_dict[df_gene.loc[df_gene.index[df_gene['Full_gene_id'] == item][0], 'gene_name']] = gene_dict[item]
#    else:
#        fin_gene_dict[item.split('.')[0]] = gene_dict[item]

In [46]:
fin_gene_dict = {}
for item in gene_dict.keys():
    idx = df_gene.index[df_gene['Full_gene_id'] == item]
    name = df_gene.loc[idx[0], 'gene_name'] if len(idx) > 0 else ''
    key = name if len(name) > 0 else item.split('.')[0]

      # keep the longest sequence when two gene IDs collide on one name
    if key in fin_gene_dict and len(fin_gene_dict[key].seq) >=len(gene_dict[item].seq):
        continue
    fin_gene_dict[key] = gene_dict[item]

In [47]:
dat.var_names

Index(['ENSMOCG00000000002', 'ENSMOCG00000000003', 'ENSMOCG00000000004',
       'ENSMOCG00000000005', 'ND1', 'ENSMOCG00000000008', 'ENSMOCG00000000009',
       'ND2', 'ENSMOCG00000000011', 'ENSMOCG00000000012',
       ...
       'ENSMOCG00000023030', 'Gemin2', 'Creb3', 'Ppfia4', 'Vps28', 'Trp53bp2',
       'Rfc1', 'ENSMOCG00000023037', 'Hbs1l', 'Celsr1'],
      dtype='object', length=19307)

In [48]:
full_set_new = set(dat.var_names) & set(fin_gene_dict.keys())

In [49]:
len(full_set_new)

18105

In [50]:
for item in fin_gene_dict.keys():
    fin_gene_dict[item].id = item
    fin_gene_dict[item].name = item

with open("../../BLASTMAPPING/Microtus_ochrogaster.MicOch1.0.cdna.curated.08292026_longestmatch.fa", "w") as handle:
    SeqIO.write(fin_gene_dict.values(), handle, "fasta") 